# บทที่ 12 การบูรณาการโครงงาน จริยธรรม และการทำซ้ำได้
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch12_capstone/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch12_capstone/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch12_capstone/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch12_capstone'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'numpy': 'numpy', 'sklearn': 'scikit-learn>=1.3'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_seed_spread.py` — โค้ดที่ 12.1

Ch.12  Which seed matters? Random-forest seed vs data-split seed on the Chapter 6 toy Higgs data.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_seed_spread.py  จากโฟลเดอร์ของบท
__file__ = '01_seed_spread.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.12  Which seed matters? Random-forest seed vs data-split seed on the Chapter 6 toy Higgs data.

Book references: ssec:repro-practices, code:seed-spread (line for line); df from Chapter 6 code:higgs-data
Numbers quoted in the text: the same seeds twice give identical AUC; forest seed only: 0.9759-0.9765 (width 0.0006);
split seed only: 0.9770-0.9820 (width ~0.005, "almost ten times").
About 1 minute.
"""
import os
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
os.chdir(HERE)
_src = (HERE.parent / 'ch06_classification' / '01_higgs_toy_data.py').read_text(encoding='utf-8')
_src = _src[_src.index('\n# --- code:higgs-data'):_src.index('\n# --- code:higgs-pipe')]
exec(_src)                                                   # make_higgs_toy, df (Chapter 6)

### โค้ดที่ 12.1

In [ ]:
# --- code:seed-spread --------------------------------------------------------
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X, y = df[['m_bb', 'pT', 'dR', 'ETmiss']].values, df['label'].values

def run(split_seed, model_seed):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y,
                                              random_state=split_seed)
    rf = RandomForestClassifier(n_estimators=100, min_samples_leaf=10,
                                random_state=model_seed, n_jobs=-1).fit(X_tr, y_tr)
    return roc_auc_score(y_te, rf.predict_proba(X_te)[:, 1])

print("same seeds twice:", run(42, 0), run(42, 0))
auc_model = [run(42, s) for s in range(10)]        # only the forest's seed changes
auc_split = [run(s, 0) for s in range(10)]         # only the data split changes
print(f"model seed: mean {np.mean(auc_model):.4f}, range "
      f"{min(auc_model):.4f}-{max(auc_model):.4f}")
print(f"split seed: mean {np.mean(auc_split):.4f}, range "
      f"{min(auc_split):.4f}-{max(auc_split):.4f}")

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    w_m, w_s = np.ptp(auc_model), np.ptp(auc_split)
    print(f"widths: model seed {w_m:.4f}, split seed {w_s:.4f} (ratio {w_s / w_m:.1f})")

## `02_manifest.py` — โค้ดที่ 12.2

Ch.12  A run manifest (versions, settings, SHA-256 of the data) and a check for duplicated rows across a split.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_manifest.py  จากโฟลเดอร์ของบท
__file__ = '02_manifest.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.12  A run manifest (versions, settings, SHA-256 of the data) and a check for duplicated rows across a split.

Book references: ssec:repro-manifest, code:manifest (line for line); continues from code:seed-spread (X, y, df)
Writes higgs_toy.csv and run_manifest.json into data/.
Numbers quoted in the text: clean split 0 overlapping rows; 300 rows recorded twice -> 121 test rows with a twin in training.
"""
import os
import sys
from pathlib import Path

import numpy as np

HERE = Path(__file__).resolve().parent
(HERE / 'data').mkdir(exist_ok=True)
_src = (HERE.parent / 'ch06_classification' / '01_higgs_toy_data.py').read_text(encoding='utf-8')
exec(_src[_src.index('\n# --- code:higgs-data'):_src.index('\n# --- code:higgs-pipe')])
_s = (HERE / '01_seed_spread.py').read_text(encoding='utf-8')
_s = _s[_s.index('\n# --- code:seed-spread'):_s.index('\ndef run(')]
exec(_s)                                                     # imports and X, y of code:seed-spread
os.chdir(HERE / 'data')

### โค้ดที่ 12.2

In [ ]:
# --- code:manifest -----------------------------------------------------------
import hashlib, json, platform, sys, datetime
import numpy, scipy, sklearn

def sha256_of(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()

def write_manifest(config, data_files, out='run_manifest.json'):
    """Record what is needed to rerun this analysis exactly."""
    manifest = {
        'created': datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'python': sys.version.split()[0],
        'platform': platform.platform(),
        'packages': {m.__name__: m.__version__ for m in (numpy, scipy, sklearn)},
        'config': config,                               # every seed and setting
        'data_sha256': {p: sha256_of(p) for p in data_files},
    }
    with open(out, 'w') as f:
        json.dump(manifest, f, indent=2)
    return manifest

def row_overlap(A, B, decimals=6):
    """Number of rows of B that also appear in A (duplicates across a split)."""
    key = lambda r: hashlib.md5(numpy.round(r, decimals).tobytes()).hexdigest()
    seen = {key(r) for r in A}
    return sum(key(r) in seen for r in B)

df.to_csv('higgs_toy.csv', index=False)                 # the exact data used
config = {'split_seed': 42, 'model_seed': 0, 'test_size': 0.3,
          'n_estimators': 100, 'min_samples_leaf': 10}
m = write_manifest(config, ['higgs_toy.csv'])
print(m['packages'], m['data_sha256']['higgs_toy.csv'][:16])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
print("overlap, clean split:", row_overlap(X_tr, X_te))
X_dup = np.vstack([X, X[:300]])                          # 300 rows recorded twice
y_dup = np.concatenate([y, y[:300]])
X_tr, X_te, _, _ = train_test_split(X_dup, y_dup, test_size=0.3, random_state=42)
print("overlap, data with duplicates:", row_overlap(X_tr, X_te))

**ส่วนตรวจผล** (ไม่อยู่ในตำรา): พิมพ์ตัวเลขที่ตำรายกมาเพื่อเทียบ

In [ ]:
# --- end of listings ---------------------------------------------------------

if __name__ == '__main__':
    X_dup = np.vstack([X, X[:300]]); y_dup = np.concatenate([y, y[:300]])
    X_tr, X_te, _, _ = train_test_split(X_dup, y_dup, test_size=0.3, random_state=42)
    print(f"expected twins in the test set if the 300 duplicates were split at random: "
          f"2 x 300 x 0.3 x 0.7 = {2 * 300 * 0.3 * 0.7:.0f}")